<a href="https://colab.research.google.com/github/acamogh/Gradio_Quiz_App/blob/main/Google_Cloud_Generative_AI_Leader_Practice_Exam_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# =====================================================================
# CELL 1: ENVIRONMENT CONFIGURATION, MOUNT DRIVE & PARSING
# =====================================================================
print("📦 Installing mobile packages... (This takes about 40 seconds)")
import sys
import subprocess
import os
import glob
import json
import random  # Imported for random chunk selection later

# Install clean standalone packages
subprocess.run([sys.executable, "-m", "pip", "install", "--ignore-installed", "-q", "-U", "groq"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "langchain-community", "pypdf", "tiktoken", "gradio"], check=True)
print("✅ Libraries installed successfully.")

import gradio as gr
from groq import Groq
from google.colab import userdata, drive
from langchain_community.document_loaders import PyPDFLoader

# Mount Google Drive to preserve logs across disconnections
print("\n🗄️ Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=True)

# Setup path to store the logs on Google Drive (Shared with Cell 2)
DRIVE_APP_DIR = "/content/drive/MyDrive/GenAI_Study_App"
os.makedirs(DRIVE_APP_DIR, exist_ok=True)
LOG_FILE_PATH = os.path.join(DRIVE_APP_DIR, "quiz_mistakes_log.json")
print(f"📁 Log Path Defined: {LOG_FILE_PATH}")

print("\n📂 Scanning notebook root for PDF files...")
pdf_files = glob.glob("*.pdf")

if not pdf_files:
    raise FileNotFoundError("⚠️ No PDF files found! Tap the folder icon on the left of Colab, upload your 8 PDFs, and run this cell again.")

print(f"📄 Found {len(pdf_files)} study guide files. Extracting content text...")

KNOWLEDGE_CHUNKS = []

for pdf in pdf_files:
    try:
        loader = PyPDFLoader(pdf)
        pages = loader.load()

        # Track page_num dynamically starting at 1 to resolve the KeyError in Cell 2
        for page_num, page in enumerate(pages, start=1):
            text = page.page_content.strip()
            if text:
                KNOWLEDGE_CHUNKS.append({
                    "source": pdf,
                    "page": page_num,
                    "content": text
                })
    except Exception as e:
        print(f"   Skipping or error reading {pdf}: {e}")

print(f"✅ Setup Complete. Loaded {len(KNOWLEDGE_CHUNKS)} individual pages into memory matrix.")

📦 Installing mobile packages... (This takes about 40 seconds)
✅ Libraries installed successfully.

🗄️ Connecting to Google Drive...
Mounted at /content/drive
📁 Log Path Defined: /content/drive/MyDrive/GenAI_Study_App/quiz_mistakes_log.json

📂 Scanning notebook root for PDF files...
📄 Found 8 study guide files. Extracting content text...
✅ Setup Complete. Loaded 588 individual pages into memory matrix.


In [28]:
# =====================================================================
# CELL 2: COMPLETE QUIZ ENGINE, DRIVER LOG MATRIX & GRADIO INTERFACE
# =====================================================================
import os
import json
import random
import gradio as gr
from groq import Groq

class MobileQuizEngine:
    def __init__(self, knowledge_chunks, log_path):
        self.chunks = knowledge_chunks
        self.log_path = log_path
        self.current_quiz = None
        self.quota_string = "🎯 RPD Left: `Pending...` | ⏳ Rolling TPM Window: `Pending...`"

        # PROD DEPLOYMENT: Llama 3.3 70B Engine configuration
        self.model_id = 'llama-3.3-70b-versatile'
        self.backup_model_id = 'llama-3.1-8b-instant'

        # Auto-restore historical weak areas directly from Google Drive path
        self.incorrect_log = self._load_history_from_drive()

        try:
            from google.colab import userdata
            api_key = userdata.get('GROQ_API_KEY')
            self.groq_client = Groq(api_key=api_key)
        except Exception:
            raise ValueError("❌ GROQ_API_KEY missing. Set it in Colab's Secrets menu (Key Icon).")

    def _load_history_from_drive(self):
        """Loads historical mistakes directly from Google Drive if available."""
        if os.path.exists(self.log_path):
            try:
                with open(self.log_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    print(f"💾 Restored {len(data)} past conceptual weak areas from Google Drive!")
                    return data
            except Exception as e:
                print(f"⚠️ Could not parse existing backup file, starting fresh: {e}")
                return []
        print("📝 No existing log found on Drive. Initializing clean slate.")
        return []

    def _write_history_to_drive(self):
        """Saves current state directly into Google Drive."""
        try:
            with open(self.log_path, 'w', encoding='utf-8') as f:
                json.dump(self.incorrect_log, f, indent=4, ensure_ascii=False)
        except Exception as e:
            print(f"🚨 Failure saving progress data to Drive: {e}")

    def generate_question(self):
        review_context = ""
        if self.incorrect_log and len(self.incorrect_log) % 3 == 0:
            past_mistake = self.incorrect_log[-1]
            review_context = f"""
            CRITICAL REVIEW CONTEXT: The student previously failed a question matching this target concept: '{past_mistake['concept']}'.
            Do NOT reuse this old question text: '{past_mistake['question_text']}'.
            Instead, heavily paraphrase the scenario, alter the variables entirely, and address the concept from a completely different angle.
            """

        # RICH CONTEXT STRATEGY: Pull 3 consecutive chunks to maintain structural topic continuity
        if len(self.chunks) >= 3:
            start_idx = random.randint(0, len(self.chunks) - 3)
            selected_chunks = self.chunks[start_idx:start_idx + 3]
        else:
            selected_chunks = self.chunks

        dynamic_context = ""
        for idx, chunk in enumerate(selected_chunks):
            dynamic_context += f"\n--- Context Source {idx+1}: File '{chunk['source']}', Page {chunk['page']} ---\n"
            dynamic_context += chunk['content'] + "\n"

        system_instruction = f"""
        You are an expert exam simulator for the Google Cloud Generative AI Leader certification.

        Knowledge Base Material with Reference Sources:
        {dynamic_context}

        Instructions:
        1. NEVER copy exact sentences or verbatim questions from the source text.
        2. Generate one highly challenging multiple-choice question matching the official certification level.
        3. Provide exactly four options (A, B, C, D) with strictly ONE correct answer.
        4. Do NOT reveal clues or answers inside your fields.
        5. Identify which specific source file and page number you primarily based the question on.

        {review_context}

        Output Structure Requirement: Return data STRICTLY inside a standard JSON object matching this structural layout:
        {{
            "question": "Clear scenario/question text goes here...",
            "options": {{
                "A": "Option text content",
                "B": "Option text content",
                "C": "Option text content",
                "D": "Option text content"
            }},
            "correct_answer": "A, B, C, or D",
            "concept": "The underlying technical principle tested",
            "explanation": "Detailed professional rationale breakdown describing why the choice is correct and others are wrong.",
            "source_file": "The exact name of the PDF file used",
            "source_page": "The exact page number used as an integer or string"
        }}
        """

        try:
            # Query Groq via raw wrapper hook to look explicitly at HTTP rate limits
            raw_response = self.groq_client.chat.completions.with_raw_response.create(
                model=self.model_id,
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": "Generate the next certification exam question based on the provided context."}
                ],
                response_format={"type": "json_object"},
                temperature=0.8
            )

            # Map remaining operational quotas straight from response headers
            headers = raw_response.headers
            rem_req = headers.get("x-ratelimit-remaining-requests", "N/A")
            rem_tok = headers.get("x-ratelimit-remaining-tokens", "N/A")
            self.quota_string = f"🧠 **Active Engine:** `{self.model_id}` | 🎯 **RPD Left:** `{rem_req}` | ⏳ **Rolling TPM Window:** `{rem_tok}`"

            completion = raw_response.parse()
            raw_content = completion.choices[0].message.content
            self.current_quiz = json.loads(raw_content)
            return self.current_quiz

        except Exception as e:
            print(f"⚡ Primary model rate-limit/timeout hit. Switching fallback... Details: {e}")
            response = self.groq_client.chat.completions.create(
                model=self.backup_model_id,
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": "Generate the next certification exam question based on the provided context."}
                ],
                response_format={"type": "json_object"},
                temperature=0.7
            )
            self.quota_string = f"⚠️ **Fallback Active:** `{self.backup_model_id}` (Live Groq quota stream bypassed)"

            try:
                raw_content = response.choices[0].message.content
                self.current_quiz = json.loads(raw_content)
                return self.current_quiz
            except Exception:
                return None

    def log_failure(self):
        if self.current_quiz:
            file_ref = self.current_quiz.get("source_file", "Unknown PDF")
            page_ref = self.current_quiz.get("source_page", "Unknown Page")

            # Store structured parameters cleanly for our UI mapping panels
            self.incorrect_log.append({
                "concept": self.current_quiz.get("concept", "General Cloud Principles"),
                "question_text": self.current_quiz.get("question", ""),
                "source_file": file_ref,
                "source_page": page_ref
            })
            self._write_history_to_drive()

    def get_summary(self):
        if not self.incorrect_log:
            return "🎉 **Perfect record! No weak areas logged yet.**"

        summary = "### 📊 Conceptual Weak Areas Dashboard\n"
        summary += "Review these specific pages in your study documentation:\n\n"
        for idx, item in enumerate(self.incorrect_log, 1):
            summary += f"{idx}. 🧠 **Concept**: `{item['concept']}`\n"
            summary += f"   📄 *File*: `{item['source_file']}` | 📖 *Page*: `{item['source_page']}`\n"
            summary += "   ---\n"
        return summary

# Instantiate active engine linkages
quiz_engine = MobileQuizEngine(KNOWLEDGE_CHUNKS, LOG_FILE_PATH)

# =====================================================================
# GRADIO INTERFACE MAPPING FUNCTIONS
# =====================================================================
def load_new_question():
    data = quiz_engine.generate_question()
    if not data:
        return "❌ API Timeout. Click 'Fetch Next Question' to retry.", [], "", "⚠️ System Error Fetching Data.", quiz_engine.quota_string

    q_text = f"### Question:\n{data['question']}"
    file_name = data.get('source_file', 'Unknown PDF')
    page_num = data.get('source_page', 'Unknown Page')

    meta_text = f"📖 **Source:** `{file_name}` (Page `{page_num}`)"
    quota_text = quiz_engine.quota_string

    choices_list = [
        f"A: {data['options'].get('A', '')}",
        f"B: {data['options'].get('B', '')}",
        f"C: {data['options'].get('C', '')}",
        f"D: {data['options'].get('D', '')}"
    ]
    return q_text, gr.Radio(choices=choices_list, value=None), "", meta_text, quota_text

def evaluate_selection(choice):
    if not quiz_engine.current_quiz:
        return "⚠️ Please load a question first."
    if not choice:
        return "⚠️ Please click an option bubble before submitting your selection."

    user_letter = choice.split(":")[0].strip().upper()
    correct_letter = quiz_engine.current_quiz['correct_answer'].strip().upper()

    if user_letter == correct_letter:
        score_text = "### ✅ CORRECT!\n\n"
    else:
        file_name = quiz_engine.current_quiz.get('source_file', 'Unknown PDF')
        page_num = quiz_engine.current_quiz.get('source_page', 'Unknown Page')
        score_text = f"### ❌ INCORRECT.\nThe correct answer choice was **({correct_letter})**.\n\n📍 *Saved to Drive! Double check this on page {page_num} of {file_name}*\n\n"
        quiz_engine.log_failure()

    return f"{score_text}📝 **Rationale Solution Breakdown**:\n{quiz_engine.current_quiz['explanation']}"

def view_summary_panel():
    return quiz_engine.get_summary()

# Custom global typography injection enforcing Google Sans font rules across elements
custom_font_css = """
@import url('https://fonts.googleapis.com/css2?family=Geist:wght@300;400;500;600;700&display=swap');

body, .gradio-container, input, button, select, textarea, span, p, h1, h2, h3, label {
    font-family: 'Google Sans', 'Product Sans', 'Geist', -apple-system, BlinkMacSystemFont, sans-serif !important;
}
h1, h2, h3 {
    font-weight: 700 !important;
}
"""

# =====================================================================
# INTERFACE CANVAS APPLICATION VIEWPORT
# =====================================================================
with gr.Blocks(theme=gr.themes.Soft(), css=custom_font_css) as mobile_app:
    gr.Markdown("# 🧠 Google Cloud GenAI Leader App Hub")

    with gr.Group():
        question_display = gr.Markdown("### Tap '➡️ Fetch Next Question' at the bottom to begin your practice run!")
        metadata_tracker = gr.Markdown("")

    choice_radio = gr.Radio(choices=[], label="👇 Select your answer option:")
    submit_btn = gr.Button("🔒 Submit Selection", variant="primary")
    feedback_display = gr.Markdown("")

    with gr.Row():
        next_btn = gr.Button("➡️ Fetch Next Question", variant="secondary")
        summary_btn = gr.Button("📊 Weak Summary Dashboard", variant="stop")

    # ISOLATED MISTAKES DASHBOARD OUTPUT TARGET
    gr.Markdown("---")
    summary_display = gr.Markdown("📊 *Click 'Weak Summary Dashboard' above to pull your performance log from Google Drive.*")

    # FIXED: Placed at the absolute bottom of the layout view canvas
    gr.Markdown("---")
    quota_display = gr.Markdown("📊 **API Quota State:** `Llama-3.3-70b-versatile` initialized. Fetch a question to update metrics.")

    # Layout routing configurations
    next_btn.click(
        fn=load_new_question,
        inputs=[],
        outputs=[question_display, choice_radio, feedback_display, metadata_tracker, quota_display],
        queue=True
    )

    submit_btn.click(fn=evaluate_selection, inputs=[choice_radio], outputs=[feedback_display])
    summary_btn.click(fn=view_summary_panel, inputs=[], outputs=[summary_display])

# Initialize runtime environment link
mobile_app.launch(inline=False, share=True, debug=False)

📝 No existing log found on Drive. Initializing clean slate.


/tmp/ipykernel_4435/601250132.py:239: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_font_css) as mobile_app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bffed24d53a082caba.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
